# Practice 2 — Predicting Product Sales

## 1. Introduction

**Objective:** Build machine learning models to predict monthly product revenue using historical sales data.

**Dataset:** UCI Online Retail (ID 352).

**Target variable:** Monthly revenue in GBP.

**Models:** Linear Regression, Decision Tree, Random Forest and Gradient Boosting.

**Evaluation:** MAE, MSE, RMSE and R².

The notebook follows the training methodology implemented in `scripts/train.py`.


## 2. Environment Setup

Import the libraries and the project's existing training module.


In [ ]:
from pathlib import Path
import os
import runpy

PROJECT = Path(r"/workspaces/Machine-Learning/PRACTICE_2_PREDICTING_PRODUCT_SALES")
SCRIPT = PROJECT / "scripts" / "train.py"
DATASET = PROJECT / "data" / "Online Retail.xlsx"

assert SCRIPT.exists(), f"Không tìm thấy train.py: {SCRIPT}"
assert DATASET.exists(), f"Không tìm thấy dataset: {DATASET}"

print("PROJECT:", PROJECT)
print("SCRIPT:", SCRIPT)
print("DATASET:", DATASET)

os.chdir(PROJECT)
runpy.run_path(str(SCRIPT), run_name="__main__")


## 3. Dataset Loading

Load the UCI Online Retail dataset using the same loader as the training pipeline.


In [ ]:
raw = training.load_data()

print("Dataset shape:", raw.shape)
display(raw.head())


## 4. Initial Data Exploration (EDA)

Inspect the columns, data types, missing values and basic statistics.


In [ ]:
print("Dataset information:")
raw.info()

print("\\nMissing values:")
display(raw.isnull().sum().to_frame("Missing values"))

print("\\nDescriptive statistics:")
display(raw.describe(include="all").T)


In [ ]:
print("Number of unique products:", raw["StockCode"].nunique())
print("Number of invoices:", raw["InvoiceNo"].nunique())
print("Countries:", raw["Country"].nunique())

if "Country" in raw.columns:
    display(raw["Country"].value_counts().head(10).to_frame("Transactions"))


## 5. Data Cleaning and Preprocessing

The project pipeline applies these rules:

- Convert invoice dates and numeric columns to appropriate types.
- Remove rows missing required fields.
- Normalize StockCode to string.
- Exclude cancelled invoices (invoice numbers beginning with C).
- Keep only positive Quantity and UnitPrice.
- Calculate revenue = Quantity × UnitPrice.


In [ ]:
clean = raw.copy()

clean["InvoiceNo"] = clean["InvoiceNo"].astype(str)
clean["InvoiceDate"] = pd.to_datetime(
    clean["InvoiceDate"], errors="coerce"
)

for column in ["Quantity", "UnitPrice"]:
    clean[column] = pd.to_numeric(
        clean[column], errors="coerce"
    )

clean = clean.dropna(
    subset=["StockCode", "InvoiceDate", "Quantity", "UnitPrice"]
)

clean["StockCode"] = clean["StockCode"].astype(str)

clean = clean[
    ~clean["InvoiceNo"].str.upper().str.startswith("C")
]

clean = clean[
    (clean["Quantity"] > 0) &
    (clean["UnitPrice"] > 0)
].copy()

clean["revenue"] = clean["Quantity"] * clean["UnitPrice"]
clean["month"] = clean["InvoiceDate"].dt.to_period("M").dt.to_timestamp()

print("Rows before cleaning:", len(raw))
print("Rows after cleaning:", len(clean))
print("Unique products after cleaning:", clean["StockCode"].nunique())

display(clean.head())


## 6. Exploratory Data Analysis

Explore monthly revenue and the distribution of transaction revenue after cleaning.


In [ ]:
monthly_revenue = (
    clean.groupby("month")["revenue"]
    .sum()
    .sort_index()
)

plt.figure(figsize=(12, 5))
monthly_revenue.plot()
plt.title("Monthly Revenue — Cleaned Transactions")
plt.xlabel("Month")
plt.ylabel("Revenue (GBP)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(clean["revenue"], bins=60)
plt.title("Distribution of Transaction Revenue")
plt.xlabel("Revenue (GBP)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
top_products = (
    clean.groupby("StockCode")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

display(top_products.to_frame("Total revenue (GBP)"))

plt.figure(figsize=(12, 5))
top_products.sort_values().plot(kind="barh")
plt.title("Top 10 Products by Total Revenue")
plt.xlabel("Revenue (GBP)")
plt.ylabel("StockCode")
plt.tight_layout()
plt.show()


## 7. Target Variable and Monthly Product Panel

The target is monthly revenue per product:

\[
Revenue = Quantity \times UnitPrice
\]

Transactions are aggregated by `StockCode` and month. Missing months between a product's first observed month and the dataset's final month are filled with zero revenue.

The project function `make_panel()` performs this processing and creates the forecasting features.


In [ ]:
panel = training.make_panel(raw)

print("Panel shape:", panel.shape)
print("Number of products:", panel["StockCode"].nunique())
print("Number of months:", panel["month"].nunique())

display(panel.head(10))


## 8. Feature Engineering

Features used by the model:

| Feature | Meaning |
|---|---|
| `StockCode` | Product identifier |
| `lag_1` | Revenue from the previous month |
| `lag_2` | Revenue from two months earlier |
| `lag_3` | Revenue from three months earlier |
| `rolling_mean_3` | Mean revenue of the previous three months |
| `month_num` | Calendar month number |

Lag and rolling features use previous months only. `shift(1)` prevents the current target from entering its own features.


In [ ]:
FEATURES = training.FEATURES

print("Features:", FEATURES)
print("Target: revenue")

display(panel[FEATURES + ["revenue", "month"]].head(10))


In [ ]:
sample_product = panel["StockCode"].iloc[0]
sample = panel[panel["StockCode"] == sample_product].sort_values("month")

plt.figure(figsize=(12, 5))
plt.plot(sample["month"], sample["revenue"], marker="o", label="Actual revenue")
plt.title(f"Monthly Revenue — Product {sample_product}")
plt.xlabel("Month")
plt.ylabel("Revenue (GBP)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 9. Chronological Train/Test Split

The data is split by month, not randomly.

- Earlier months are used for training.
- Later months are reserved for testing.
- No shuffling is applied.

This simulates forecasting future revenue from historical observations.


In [ ]:
months = np.sort(panel["month"].unique())

if len(months) < 8:
    raise ValueError("At least 8 months are required after preprocessing.")

split_idx = max(1, int(len(months) * 0.8))
test_start = months[split_idx]

train = panel[panel["month"] < test_start].copy()
test = panel[panel["month"] >= test_start].copy()

X_train = train[FEATURES]
y_train = train["revenue"]

X_test = test[FEATURES]
y_test = test["revenue"]

print("Training rows:", len(train))
print("Testing rows:", len(test))
print("Training period:", train["month"].min(), "to", train["month"].max())
print("Testing starts:", test_start)


In [ ]:
split_summary = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "Rows": [len(train), len(test)],
    "Start": [train["month"].min(), test["month"].min()],
    "End": [train["month"].max(), test["month"].max()]
})

display(split_summary)


## 10. Model Pipeline

The preprocessing pipeline:

- Numeric features: median imputation.
- Product identifier: OneHotEncoder.
- Estimator: the selected regression algorithm.

The implementation reuses `build_pipeline()` from `scripts/train.py`.


In [ ]:
candidates = {
    "Linear Regression": training.build_pipeline(
        LinearRegression()
    ),
    "Decision Tree": training.build_pipeline(
        DecisionTreeRegressor(random_state=training.SEED)
    ),
    "Random Forest": training.build_pipeline(
        RandomForestRegressor(
            random_state=training.SEED,
            n_jobs=1
        )
    ),
    "Gradient Boosting": training.build_pipeline(
        GradientBoostingRegressor(
            random_state=training.SEED
        )
    )
}

print("Candidate models:")
for name in candidates:
    print("-", name)


## 11. Time Series Cross-Validation

Use `TimeSeriesSplit` on the training months.

All products belonging to the same month remain in the same fold. This avoids splitting products from one month across training and validation subsets.

The selection metric is negative RMSE in scikit-learn; we convert it to positive RMSE for interpretation. Lower RMSE is better.


In [ ]:
n_splits = min(
    3,
    max(2, len(np.unique(train["month"])) // 3)
)

month_cv = TimeSeriesSplit(n_splits=n_splits)
train_months = np.sort(train["month"].unique())

cv = [
    (
        np.flatnonzero(train["month"].isin(train_months[tr_idx])),
        np.flatnonzero(train["month"].isin(train_months[va_idx]))
    )
    for tr_idx, va_idx in month_cv.split(train_months)
]

print("Number of CV folds:", len(cv))

for i, (tr_idx, va_idx) in enumerate(cv, start=1):
    print(
        f"Fold {i}: train rows={len(tr_idx)}, "
        f"validation rows={len(va_idx)}"
    )


## 12. Train and Compare Baseline Models

Each candidate is evaluated using the same chronological CV folds. The models are then fitted on the complete training set.

The test set is not used for model selection.


In [ ]:
cv_scores = {}
fitted = {}

for name, pipe in candidates.items():
    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=2
    )

    cv_scores[name] = float(-scores.mean())

    pipe.fit(X_train, y_train)
    fitted[name] = pipe

    print(
        f"{name}: CV RMSE = {cv_scores[name]:.4f}"
    )


## 13. Random Forest Hyperparameter Tuning

RandomizedSearchCV searches for a better Random Forest configuration.

Tuning is performed only on training data and uses the same time-series CV folds.


In [ ]:
search = RandomizedSearchCV(
    training.build_pipeline(
        RandomForestRegressor(
            random_state=training.SEED,
            n_jobs=1
        )
    ),
    {
        "model__n_estimators": [100, 200],
        "model__max_depth": [8, 12, None],
        "model__min_samples_leaf": [2, 4],
        "model__max_features": [0.8, 1.0]
    },
    n_iter=6,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=training.SEED,
    n_jobs=2,
    verbose=1,
    refit=True
)

search.fit(X_train, y_train)

tuned_name = "Random Forest (tuned)"
fitted[tuned_name] = search.best_estimator_
cv_scores[tuned_name] = float(-search.best_score_)

print("Best parameters:")
print(search.best_params_)
print("Best CV RMSE:", cv_scores[tuned_name])


## 14. Model Selection

Select the model with the lowest mean CV RMSE on the training data.

The independent test set is not used to choose the model.


In [ ]:
best_name = min(cv_scores, key=cv_scores.get)
best_model = fitted[best_name]

cv_table = (
    pd.DataFrame([
        {"Model": name, "CV_RMSE": score}
        for name, score in cv_scores.items()
    ])
    .sort_values("CV_RMSE")
    .reset_index(drop=True)
)

display(cv_table)

print("Selected model:", best_name)


## 15. Final Evaluation on the Test Set

After model selection, evaluate each fitted model on the held-out test period.

Metrics:
- **MAE:** Mean Absolute Error.
- **MSE:** Mean Squared Error.
- **RMSE:** Root Mean Squared Error.
- **R²:** Coefficient of determination.

All errors are measured on the original revenue scale (GBP).


In [ ]:
test_rows = []

for name, pipe in fitted.items():
    predictions = np.maximum(0, pipe.predict(X_test))

    metrics = training.evaluate(y_test, predictions)

    test_rows.append({
        "Model": name,
        **metrics,
        "CV_RMSE": cv_scores[name]
    })

comparison = (
    pd.DataFrame(test_rows)
    .sort_values("CV_RMSE")
    .reset_index(drop=True)
)

display(comparison)


In [ ]:
best_predictions = np.maximum(
    0,
    best_model.predict(X_test)
)

best_metrics = training.evaluate(
    y_test,
    best_predictions
)

print("Selected model:", best_name)
print("Test metrics:")
for metric, value in best_metrics.items():
    print(f"{metric}: {value:.4f}")


## 16. Visualization of Model Performance

The following charts compare test RMSE and the actual versus predicted revenue of the selected model.

Remember: test metrics are for final evaluation, not model selection.


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(comparison["Model"], comparison["RMSE"])
plt.title("Test RMSE by Model")
plt.ylabel("RMSE (GBP)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, best_predictions, alpha=0.35)

max_value = max(float(y_test.max()), float(best_predictions.max()))
plt.plot([0, max_value], [0, max_value], linestyle="--")

plt.title(f"Actual vs Predicted — {best_name}")
plt.xlabel("Actual revenue (GBP)")
plt.ylabel("Predicted revenue (GBP)")
plt.tight_layout()
plt.show()


In [ ]:
residuals = y_test.to_numpy() - best_predictions

plt.figure(figsize=(10, 5))
plt.hist(residuals, bins=50)
plt.title(f"Residual Distribution — {best_name}")
plt.xlabel("Residual (Actual - Predicted), GBP")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## 17. Results Interpretation

Use the actual outputs above to discuss:

1. Which model has the lowest CV RMSE?
2. Why was that model selected?
3. Does the selected model also have the lowest test RMSE?
4. Is there a large difference between CV RMSE and test RMSE?
5. What might explain a performance difference between validation and test periods?

Do not claim that the model is highly accurate without considering the test metrics and limitations.


In [ ]:
print("Selected by CV:", best_name)
print("CV RMSE:", cv_scores[best_name])
print("Test RMSE:", best_metrics["RMSE"])
print("Test MAE:", best_metrics["MAE"])
print("Test R2:", best_metrics["R2"])

print("\\nBest Random Forest parameters:")
print(search.best_params_)


## 18. Limitations

- Sales patterns may change over time.
- Some products have limited historical observations.
- Missing product-month revenue is treated as zero.
- The model uses a limited set of lag and calendar features.
- The difference between cross-validation and test performance may indicate a change in the data distribution.
- The model predicts monthly product revenue, not individual transaction quantities.

## 19. Conclusion

This project builds a monthly product revenue forecasting pipeline using the UCI Online Retail dataset.

It compares four regression algorithms, applies time-aware cross-validation, tunes Random Forest and selects the final model using CV RMSE.

The final test period is kept separate for independent evaluation.

**Final selected model and metrics must be interpreted from the executed notebook outputs.**


## Appendix — Reproducibility

The notebook reuses the project's `scripts/train.py` functions for data loading, panel construction, preprocessing and evaluation.

To regenerate the saved model and project output files, run the training script separately from the project directory:

```bash
python scripts/train.py
```

This notebook itself does not overwrite the saved model or output files.
